In [1]:
import socket
import cv2
import numpy as np
import time
import math

from pynq.overlays.base import BaseOverlay
base = BaseOverlay("base.bit")
leds = base.leds

### Functions

In [2]:
def bgr_to_lab(color):
    bgr = np.array(color, dtype=np.uint8).reshape(1,1,3)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    return tuple(lab[0,0])

In [3]:
OFFSET = 140

def chroma(a,b):
    return np.sqrt((a-OFFSET)**2 + (b-OFFSET)**2)

def determine_season_lab(colors):
    skin, hair, eye = colors

    sL,sA,sB = skin
    hL,hA,hB = hair
    eL,eA,eB = eye

    # hue (warm vs cool)
    warm_score = (sA-OFFSET) + (sB-OFFSET)
    print(f"warmth: {warm_score}")
    hue = "warm" if warm_score > 0 else "cool"

    # chroma (bright vs muted)
    avg_chroma = (0.6*chroma(sA,sB) +
                  0.3*chroma(hA,hB) +
                  0.1*chroma(eA,eB)) / 3

    chroma_type = "bright" if avg_chroma >= 10 else "muted"
    print(f"chroma avg: {avg_chroma}")

    # value contrast (high vs low)
    contrast = max(sL,hL,eL) - min(sL,hL,eL)


    print(f"hue: {hue}, contrast: {contrast}, chroma: {chroma_type}")

    if hue == "cool":
        if int(contrast) > 45:
            season = "winter"
        elif int(contrast) < 35:
            season = "summer"
        else:
            if chroma_type == "bright":
                season = "winter"
            else: # chroma = muted
                season = "summer"
    else: # hue == "warm":
        if chroma_type == "bright":
            season = "spring"
        else: # chroma == muted
            season = "autumn"
    
    return season

In [4]:
 def all_onboard_leds_off():
    for a in leds:
        a.off()

### Main

In [5]:
# input is skin, hair, eye in BGR format
colors = []
all_onboard_leds_off()

CLIENT_IP = "0.0.0.0"
LISTENING_PORT = 9999

sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

sock.bind((CLIENT_IP, LISTENING_PORT))

sock.listen(1)

conn, addr = sock.accept()
print(f"Accepted connection from {addr}")

while True:
    data = conn.recv(1024)
    print("data comming in")
    if not data:
        print("Client disconneted")
        break
    print("Received message: ", data.decode('utf8'))
    bgr_tuple = tuple(map(int, data.split()))
    colors.append(bgr_tuple)

print("all data received") 
sock.shutdown(socket.SHUT_RDWR)
sock.close()

print(f"bgr colors: {colors}")

# skin, hair, eye in lab format
lab_colors = []
for color in colors:
    lab_colors.append(bgr_to_lab(color))
print(f"lab: {lab_colors}")

print("begin analysis")
season = determine_season_lab(lab_colors)
print(season)

led_num = -1

# solid color button logic
if season == 'spring':
    led_num = 0
elif season == 'summer':
    led_num = 1
elif season == 'autumn':
    led_num = 2
elif season == 'winter':
    led_num = 3
else: pass

for i in range(4):
    leds[led_num].on()
    time.sleep(0.5)
    leds[led_num].off()
    time.sleep(0.5)

leds[led_num].on()

# TODO: colorfull led stuff

print("Done.")

Accepted connection from ('192.168.2.1', 22839)



KeyboardInterrupt

